In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from pathlib import Path

# Folder kerja notebook saat ini
DATA_DIR = Path.cwd()

# Lokasi file embedding + metadata hasil export fasttext.ipynb
EMB_DIR = DATA_DIR / 'Embedding' / '2'
META_DIR = EMB_DIR

def require_file(path):
    if not path.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {path}")
    return path

# Load file sesuai format simpan
answers_path = require_file(EMB_DIR / 'answers_emb.npy')
questions_path = require_file(EMB_DIR / 'questions_emb.npy')
answerkeys_path = require_file(EMB_DIR / 'answerkeys_emb.npy')
pkl_path = require_file(META_DIR / 'metadata.pkl')

answers_emb = np.load(answers_path)
questions_emb_unique = np.load(questions_path)
answerkeys_emb_unique = np.load(answerkeys_path)
metadata = pd.read_pickle(pkl_path).copy()

# Wajib ada psj_idx karena questions/answerkeys disimpan per IDPSJ unik
if 'psj_idx' not in metadata.columns:
    raise KeyError("Kolom 'psj_idx' tidak ditemukan di metadata.pkl")

psj_idx = metadata['psj_idx'].astype(int).to_numpy()
if psj_idx.min() < 0 or psj_idx.max() >= len(questions_emb_unique):
    raise ValueError(
        "Nilai psj_idx di metadata di luar range questions/answerkeys embedding."
    )

# Rekonstruksi ke level per sampel agar panjang sejajar dengan answers_emb
questions_emb = questions_emb_unique[psj_idx]
answerkeys_emb = answerkeys_emb_unique[psj_idx]

# Samakan panjang metadata dengan answers jika ada mismatch minor
n = min(len(metadata), len(answers_emb))
metadata = metadata.iloc[:n].copy().reset_index(drop=True)
answers_emb = answers_emb[:n]
questions_emb = questions_emb[:n]
answerkeys_emb = answerkeys_emb[:n]

print("=== Hasil Load (Format Simpan Konsisten) ===")
print(f"answers_emb (per sampel)         : {answers_emb.shape} ({answers_path})")
print(f"questions_emb unik (per IDPSJ)   : {questions_emb_unique.shape} ({questions_path})")
print(f"answerkeys_emb unik (per IDPSJ)  : {answerkeys_emb_unique.shape} ({answerkeys_path})")
print(f"metadata.pkl                     : {pkl_path}")
print(f"\nMetadata rows                   : {len(metadata)}")
print(f"Kolom metadata                  : {list(metadata.columns)}")
print(f"IDPSJ unik                      : {metadata['IDPSJ'].astype(str).nunique()}")

# Parsing grade supaya konsisten untuk analisis distribusi
metadata['grade_num'] = (
    metadata['grade'].astype(str)
    .str.replace(',', '.', regex=False)
    .astype(float)
    .round()
    .astype(int)
)

print(f"\nDistribusi grade:")
print(metadata['grade_num'].value_counts().sort_index())
print(f"\nArray siap pakai per sampel:")
print(f"questions_emb  : {questions_emb.shape}")
print(f"answerkeys_emb : {answerkeys_emb.shape}")
print(f"answers_emb    : {answers_emb.shape}")

In [ ]:
# ── Analisis Distribusi per IDPSJ untuk Persiapan Mixup/SMOTE ─────────────────

# Pivot: baris=IDPSJ, kolom=grade_num, nilai=jumlah sampel
pivot = (metadata.groupby(['IDPSJ', 'grade_num'])
                 .size()
                 .unstack(fill_value=0)
                 .reindex(columns=range(1, 11), fill_value=0))

print("=" * 70)
print("Jumlah sampel per grade per IDPSJ")
print("=" * 70)
print(pivot.to_string())

# Statistik ringkas per IDPSJ
stats = pd.DataFrame({
    'total'   : pivot.sum(axis=1),
    'max_cls' : pivot.max(axis=1),
    'min_cls' : pivot[pivot > 0].min(axis=1),
    'n_kelas' : (pivot > 0).sum(axis=1),
})
stats['median_cls'] = pivot.replace(0, np.nan).median(axis=1)
stats['target_1:2'] = (stats['max_cls'] / 2).apply(np.ceil).astype(int)
stats['target_1:3'] = (stats['max_cls'] / 3).apply(np.ceil).astype(int)

print("\n" + "=" * 70)
print("Statistik per IDPSJ")
print("=" * 70)
print(stats.to_string())

# Ringkasan global
global_max = pivot.max().max()
global_target_half = int(np.ceil(global_max / 2))
print(f"\nNilai kelas terbanyak secara global : {global_max}")
print(f"Target rasio 1:2 (max/2)            : {global_target_half}")
print(f"Target flat 10                      : 10")
print(f"\nKelas yang AKAN di-augmentasi (< target 1:2) per IDPSJ:")
for psj in pivot.index:
    row = pivot.loc[psj]
    local_max = row.max()
    tgt = int(np.ceil(local_max / 2))
    kurang = row[(row > 0) & (row < tgt)]
    if not kurang.empty:
        detail = ", ".join([f"grade {g}:{cnt}→{tgt}" for g, cnt in kurang.items()])
        print(f"  IDPSJ {psj} (target={tgt}): {detail}")

In [ ]:
# Gunakan metadata + embedding yang sudah di-load dari metadata.pkl (format final)
base_df = metadata.copy().reset_index(drop=True)

# Samakan panjang agar aman jika ada perbedaan minor
n = min(len(base_df), len(answers_emb), len(answerkeys_emb), len(questions_emb))
base_df = base_df.iloc[:n].copy().reset_index(drop=True)
answers_base = answers_emb[:n]
answerkeys_base = answerkeys_emb[:n]
questions_base = questions_emb[:n]

# Pastikan grade_num tersedia
if 'grade_num' not in base_df.columns:
    base_df['grade_num'] = (
        base_df['grade'].astype(str)
        .str.replace(',', '.', regex=False)
        .astype(float)
        .round()
        .astype(int)
    )

print(f"Rows aligned: {n}")
print(f"IDPSJ unik  : {sorted(base_df['IDPSJ'].astype(str).unique())}")
print("Distribusi grade (base):")
print(base_df['grade_num'].value_counts().sort_index())

In [ ]:
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
TARGET_RATIO = 0.5  # target minoritas = ceil(max_kelas * 0.5)

all_syn_emb = []
all_syn_meta = []

seq_shape = answers_base.shape[1:]  # contoh: (80, 300)

for psj, g in base_df.groupby('IDPSJ', sort=True):
    idx = g.index.to_numpy()

    # SMOTE butuh 2D: flatten sequence embedding
    X_seq = answers_base[idx]
    X = X_seq.reshape(len(X_seq), -1)
    y = g['grade_num'].to_numpy()

    cls_counts = pd.Series(y).value_counts().sort_index()
    if cls_counts.size < 2:
        continue

    local_max = int(cls_counts.max())
    target = int(np.ceil(local_max * TARGET_RATIO))

    sampling_strategy = {
        int(c): target
        for c, cnt in cls_counts.items()
        if (cnt >= 2) and (cnt < target)
    }
    if not sampling_strategy:
        continue

    min_count_targeted = min(int(cls_counts[c]) for c in sampling_strategy.keys())
    k_neighbors = max(1, min(5, min_count_targeted - 1))

    sm = SMOTE(
        sampling_strategy=sampling_strategy,
        k_neighbors=k_neighbors,
        random_state=RANDOM_STATE,
    )
    X_res, y_res = sm.fit_resample(X, y)

    n_new = len(X_res) - len(X)
    if n_new <= 0:
        continue

    X_syn_flat = X_res[-n_new:]
    y_syn = y_res[-n_new:]

    # Mapping ke sampel asli terdekat dalam grup (IDPSJ + grade) untuk menjaga konteks
    X_norm = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)

    for vec_flat, gy in zip(X_syn_flat, y_syn):
        same_grade_local = np.where(y == gy)[0]
        if len(same_grade_local) == 0:
            same_grade_local = np.arange(len(y))

        v = vec_flat / (np.linalg.norm(vec_flat) + 1e-12)
        sim = X_norm[same_grade_local] @ v
        local_best = same_grade_local[int(np.argmax(sim))]
        global_best_idx = idx[local_best]

        all_syn_emb.append(vec_flat.reshape(seq_shape))
        all_syn_meta.append({
            'IDPSJ': str(psj),
            'grade_num': int(gy),
            'nearest_idx': int(global_best_idx),
        })

if all_syn_emb:
    syn_emb = np.array(all_syn_emb, dtype=np.float32)
else:
    syn_emb = np.empty((0, *seq_shape), dtype=np.float32)

syn_meta = pd.DataFrame(all_syn_meta)

print(f"Synthetic samples (answers_emb): {len(syn_emb)}")
if len(syn_meta):
    print("Distribusi synthetic per grade:")
    print(syn_meta['grade_num'].value_counts().sort_index())

In [ ]:
from gensim.models.fasttext import load_facebook_vectors

FT_PATH = Path('Pre-trained FastText Model') / 'cc.id.300.bin'
if not FT_PATH.exists():
    raise FileNotFoundError(f"FastText model tidak ditemukan: {FT_PATH}")

print('Load FastText vectors (ini bisa agak lama)...')
ft = load_facebook_vectors(str(FT_PATH))

In [ ]:
def inverse_words_from_vec(vec_300, model, topn=12):
    vec_300 = vec_300.astype(np.float32)
    vec_300 = vec_300 / (np.linalg.norm(vec_300) + 1e-12)
    return [w for w, _ in model.similar_by_vector(vec_300, topn=topn)]


if len(syn_meta) > 0:
    inv_words = []
    recon_text = []

    has_answer_col = 'answer' in base_df.columns

    for i, row in syn_meta.iterrows():
        seq_vec = syn_emb[i]  # shape: (seq_len, 300)

        # Pooling sequence -> 300 dim agar bisa dipetakan ke vocab FastText
        sent_vec = seq_vec.mean(axis=0)
        words = inverse_words_from_vec(sent_vec, ft, topn=12)
        inv_words.append(' '.join(words))

        nn_idx = int(row['nearest_idx'])
        if has_answer_col:
            recon_text.append(str(base_df.loc[nn_idx, 'answer']))
        else:
            recon_text.append('')

    syn_meta['inverse_words'] = inv_words
    syn_meta['answer_reconstructed'] = recon_text

    # Turunkan metadata synthetic dari nearest sample agar kolom tetap konsisten
    syn_full_meta = base_df.iloc[syn_meta['nearest_idx'].astype(int)].copy().reset_index(drop=True)
    syn_full_meta['grade_num'] = syn_meta['grade_num'].values
    syn_full_meta['grade'] = syn_meta['grade_num'].astype(str).values
    if 'answer' in syn_full_meta.columns:
        syn_full_meta['answer'] = syn_meta['answer_reconstructed'].values
    syn_full_meta['inverse_words'] = syn_meta['inverse_words'].values
    syn_full_meta['is_synthetic'] = 1
else:
    syn_full_meta = base_df.iloc[0:0].copy()
    syn_full_meta['inverse_words'] = []
    syn_full_meta['is_synthetic'] = []

base_out = base_df.copy()
base_out['is_synthetic'] = 0
base_out['inverse_words'] = ''

aug_meta = pd.concat([base_out, syn_full_meta], ignore_index=True)

# Embedding untuk questions/answerkeys synthetic diambil dari nearest sample
if len(syn_meta) > 0:
    nn_idx_arr = syn_meta['nearest_idx'].astype(int).to_numpy()
    syn_q = questions_base[nn_idx_arr]
    syn_ak = answerkeys_base[nn_idx_arr]
else:
    syn_q = np.empty((0, *questions_base.shape[1:]), dtype=np.float32)
    syn_ak = np.empty((0, *answerkeys_base.shape[1:]), dtype=np.float32)

aug_answers = np.vstack([answers_base, syn_emb])
aug_questions = np.vstack([questions_base, syn_q])
aug_answerkeys = np.vstack([answerkeys_base, syn_ak])

np.save('smote_answers_emb.npy', aug_answers)
np.save('smote_questions_emb.npy', aug_questions)
np.save('smote_answerkeys_emb.npy', aug_answerkeys)
aug_meta.to_csv('smote_datafix.csv', index=False)

print('\n=== RINGKASAN OUTPUT ===')
print(f"Original samples  : {len(base_df)}")
print(f"Synthetic samples : {len(syn_meta)}")
print(f"Total samples     : {len(aug_meta)}")
print('Saved: smote_answers_emb.npy, smote_questions_emb.npy, smote_answerkeys_emb.npy, smote_datafix.csv')

preview_cols = [c for c in ['IDPSJ', 'grade', 'is_synthetic', 'inverse_words', 'answer'] if c in aug_meta.columns]
display(aug_meta[preview_cols].tail(10))